# Prétraitement des données

Ce notebook prépare le jeu de données pour la phase de modélisation. Il supprime
les colonnes non exploitables, sépare les variables explicatives de la cible,
encode les variables catégorielles et sauvegarde un jeu de données prêt à
l'emploi.

In [1]:
# Import des bibliothèques
import pandas as pd
import numpy as np

In [2]:
# Chargement des données
df = pd.read_csv(r"C:\Users\hp\Desktop\data-quality-bank\data\ml\ml_dataset_v2.csv")
df.shape

(505500, 12)

## Suppression des colonnes non retenues

Trois catégories de colonnes sont supprimées :
- **Fuite de données** (`is_valide`, `decision_type`, `dq_score`) : dérivées de la
  décision, elles révèlent la réponse.
- **Redondantes avec `rule_id`** (`dimension`, `severity`) : entièrement déduites de
  la règle, elles n'apportent aucune information supplémentaire.
- **Identifiants sans valeur prédictive** (`record_id`, `date_id`).

Les variables conservées sont `table_origine`, `rule_id`, `contexte_reporting` et
`valeur_metier`, ainsi que la cible `label`.

In [3]:
colonnes_a_supprimer = [
    "is_valide", "decision_type", "dq_score",   # fuite de données
    "dimension", "severity",                     # redondantes avec rule_id
    "record_id", "date_id"                       # identifiants
]

df_clean = df.drop(columns=colonnes_a_supprimer)
df_clean.head()

,table_origine,rule_id,label,contexte_reporting,valeur_metier
0,clients,CONTACT_INVALIDE,rejected,CAMPAGNE_CONTACT,0.00
1,transactions,TX_NON_GAB_AVEC_ID,accepted,CAMPAGNE_CONTACT,10700.08
2,transactions,TX_NON_GAB_AVEC_ID,accepted,CAMPAGNE_CONTACT,6001.62
3,transactions,TX_NON_GAB_AVEC_ID,accepted,CAMPAGNE_CONTACT,2103.28
4,transactions,TX_NON_GAB_AVEC_ID,accepted,CAMPAGNE_CONTACT,19597.66


In [ ]:
# Vérification des colonnes restantes :
df_clean.columns

Index(['table_origine', 'rule_id', 'label', 'contexte_reporting',
       'valeur_metier'],
      dtype='object')

## Séparation des variables explicatives et de la cible

`X` regroupe les variables explicatives (`table_origine`, `rule_id`,
`contexte_reporting`, `valeur_metier`).
`y` correspond à la cible (`label`).

In [5]:
X = df_clean.drop(columns=["label"])
y = df_clean["label"]

print("Features :", list(X.columns))
print("Cible    :", y.name)

Features : ['table_origine', 'rule_id', 'contexte_reporting', 'valeur_metier']
Cible    : label


## Encodage de la cible

La cible est encodée en valeurs numériques. La classe positive (1) correspond aux
enregistrements **rejetés**, c'est-à-dire la classe minoritaire que l'on cherche à
détecter. Ce choix rend les métriques d'évaluation (précision, rappel, F1)
directement interprétables sur les rejets.

In [6]:
y_encoded = y.map({"accepted": 0, "rejected": 1})

# Vérification de la répartition
print(y_encoded.value_counts())
print()
print(y_encoded.value_counts(normalize=True) * 100)

label
0    476860
1     28640
Name: count, dtype: int64

label
0    94.334322
1     5.665678
Name: proportion, dtype: float64


## Encodage des variables explicatives (one-hot encoding)

Les variables `rule_id`, `table_origine` et `contexte_reporting` sont catégorielles
nominales (sans ordre naturel). Le one-hot encoding crée une colonne binaire par
catégorie, évitant ainsi d'introduire un ordre artificiel entre les valeurs. La
variable `valeur_metier`, étant numérique, est conservée telle quelle.

In [10]:
X_encoded = pd.get_dummies(X, columns=["rule_id", "table_origine", "contexte_reporting"])

print("Dimensions après encodage :", X_encoded.shape)
X_encoded.head(-1)

Dimensions après encodage : (505500, 18)


,valeur_metier,rule_id_AUCUNE,rule_id_CARTE_ACTIVE_COMPTE_CLOTURE,rule_id_CLIENT_AGENCE_FERMEE,rule_id_COMPTE_CLIENT_INEXISTANT,rule_id_CONTACT_INVALIDE,rule_id_DIGITAL_CLIENT_INEXISTANT,rule_id_TX_COMPTE_INEXISTANT,rule_id_TX_GAB_SANS_ID,rule_id_TX_NON_GAB_AVEC_ID,table_origine_cartes,table_origine_clients,table_origine_comptes,table_origine_digital_usage,table_origine_transactions,contexte_reporting_ANALYSE_GEOGRAPHIQUE,contexte_reporting_ANALYSE_TRANSACTIONNELLE,contexte_reporting_CAMPAGNE_CONTACT
0,0.00,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True
1,10700.08,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,True
2,6001.62,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,True
3,2103.28,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,True
4,19597.66,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
505494,0.00,True,False,False,False,False,False,False,False,False,False,False,False,False,True,True,False,False
505495,0.00,True,False,False,False,False,False,False,False,False,False,False,False,False,True,True,False,False
505496,0.00,True,False,False,False,False,False,False,False,False,False,False,False,False,True,True,False,False
505497,0.00,True,False,False,False,False,False,False,False,False,False,False,False,False,True,True,False,False


## Vérification de la dimensionnalité

Le rapport entre le nombre de lignes et le nombre de colonnes est vérifié afin de
s'assurer de l'absence de problème de dimensionnalité. Un rapport élevé garantit que
le modèle dispose de suffisamment d'exemples par variable.

In [8]:
n_lignes, n_colonnes = X_encoded.shape
print(f"Nombre de lignes         : {n_lignes}")
print(f"Nombre de colonnes       : {n_colonnes}")
print(f"Rapport lignes/colonnes  : {n_lignes / n_colonnes:.0f} lignes par colonne")

Nombre de lignes         : 505500
Nombre de colonnes       : 18
Rapport lignes/colonnes  : 28083 lignes par colonne


## Sauvegarde du jeu de données prétraité

Le jeu de données final, regroupant les variables encodées et la cible, est
sauvegardé pour la phase de modélisation.

Remarque : le rééchantillonnage destiné à corriger le déséquilibre des classes
n'est **pas** effectué ici. Il sera appliqué uniquement sur les données
d'entraînement, après le découpage, afin d'éviter toute fuite de données vers
l'ensemble de test.

In [9]:
df_final = X_encoded.copy()
df_final["label"] = y_encoded.values

chemin_sortie = r"C:\Users\hp\Desktop\data-quality-bank\data\ml\fct_quality_preprocessed.csv"
df_final.to_csv(chemin_sortie, index=False)

print("Dataset prétraité sauvegardé.")
print("Dimensions finales :", df_final.shape)
df_final.head()

Dataset prétraité sauvegardé.
Dimensions finales : (505500, 19)


,valeur_metier,rule_id_AUCUNE,rule_id_CARTE_ACTIVE_COMPTE_CLOTURE,rule_id_CLIENT_AGENCE_FERMEE,rule_id_COMPTE_CLIENT_INEXISTANT,rule_id_CONTACT_INVALIDE,rule_id_DIGITAL_CLIENT_INEXISTANT,rule_id_TX_COMPTE_INEXISTANT,rule_id_TX_GAB_SANS_ID,rule_id_TX_NON_GAB_AVEC_ID,table_origine_cartes,table_origine_clients,table_origine_comptes,table_origine_digital_usage,table_origine_transactions,contexte_reporting_ANALYSE_GEOGRAPHIQUE,contexte_reporting_ANALYSE_TRANSACTIONNELLE,contexte_reporting_CAMPAGNE_CONTACT,label
0,0.00,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,1
1,10700.08,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,True,0
2,6001.62,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,True,0
3,2103.28,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,True,0
4,19597.66,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,True,0
